# Mycol vs manual germination scoring

Working behind Supplementary Figure 8 and Figure 3f, covering what neither figure draws.

- **Data** — three raters scored the same 120 images twice, by hand and with Mycol
- **Gives** — ICC(C,1) between raters · counting time and the paired t-test Figure 3f reports ·
  per-image mean +/- SD, ranked by count
- **Not here** — the rater-pair difference chart *is* Supplementary Figure 8, drawn by `../figureS8.ipynb`
- **Kernel** — `mycol_colonies_env`, run top to bottom


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pingouin as pg
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

XLSX = Path("..") / "assets" / "20250908_All_data_manual_vs_tool.xlsx"
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

# One sheet per rater x method. The sheets are named after the raters; they are relabelled
# User1/2/3 here, and disagree on column names ("Photo" vs "Image", mixed case, trailing
# spaces), so the columns are normalised on the way in.
SHEETS = {("User1", "manual"): "Alex_manual", ("User2", "manual"): "Gio_manual",
          ("User3", "manual"): "Rafa_manual", ("User1", "tool"): "Alex_tool",
          ("User2", "tool"): "Gio_tool", ("User3", "tool"): "Rafa_tool"}


def read_sheet(sheet, user, method):
    df = pd.read_excel(XLSX, sheet_name=sheet)
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns={"photo": "image", "number": "image_number"})
    return df[["image_number", "germinated", "ungerminated"]].assign(user=user, method=method)


combined = pd.concat([read_sheet(sheet, u, m) for (u, m), sheet in SHEETS.items()],
                     ignore_index=True)
METHOD_LABEL = {"manual": "Manual", "tool": "Mycol"}

print(f"{len(combined)} rows: {combined['image_number'].nunique()} images x 3 raters x 2 methods")
combined.head()


## Inter-class correlation

**ICC(C,1)** - two-way mixed, consistency, single measures: the raters are fixed people not a sample,
rater-level offsets are allowed (ranking matters, not identical absolute numbers), and the value
describes one rater rather than the average of three.


In [ ]:
rows = []
for method in ("manual", "tool"):
    for count in ("germinated", "ungerminated"):
        r = pg.intraclass_corr(data=combined[combined["method"] == method],
                               targets="image_number", raters="user", ratings=count
                               ).set_index("Type").loc["ICC(C,1)"]
        rows.append({"method": METHOD_LABEL[method], "count": count,
                     "ICC": round(r["ICC"], 3),
                     "CI_low": round(r["CI95"][0], 3), "CI_high": round(r["CI95"][1], 3),
                     "p": round(r["pval"], 4)})

icc = pd.DataFrame(rows)
print("ICC(C,1) - two-way mixed, consistency, single measures:")
print(icc.to_string(index=False))


## Counting time

Normalised to each rater's own manual time (= 100%), since baseline speeds differ. The Mycol bar
splits into hands-on counting plus unattended processing; the paired t-test is the one Figure 3f reports.


In [ ]:
t = (pd.read_excel(XLSX, sheet_name="time")
       .rename(columns={"Unnamed: 0": "User", "Manual ": "Manual"}))
t = t[t["User"].isin(["User1", "User2", "User3"])].copy()
t["count_pct"] = t["Mycol Assisted"] / t["Manual"] * 100
t["proc_pct"] = t["Processing Time"] / t["Manual"] * 100

total = (t["count_pct"] + t["proc_pct"]).to_numpy()
t_stat, p_val = stats.ttest_rel(np.full(len(total), 100.0), total)
sd = float(np.std(total, ddof=1))
sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"

print(f"paired t-test n={len(total)}: t = {t_stat:.3f}, p = {p_val:.4f}  ->  {sig}")
print(f"Mycol total {total.mean():.1f}% of manual (SD {sd:.1f})"
      f"  ->  {100 - total.mean():.1f}% less time")

# a "Mean" group on the right, carrying the across-rater SD; manual has no error bar
# because every rater's manual time is 100% by construction
x = list(t["User"]) + ["Mean"]
counting = list(t["count_pct"]) + [t["count_pct"].mean()]
processing = list(t["proc_pct"]) + [t["proc_pct"].mean()]
edge = dict(marker_line_color="black", marker_line_width=1)

fig_time = go.Figure([
    go.Bar(name="Manual", x=x, y=[100.0] * 4, offsetgroup=0, marker_color="#a8d5a2", **edge),
    go.Bar(name="Mycol counting", x=x, y=counting, offsetgroup=1,
           marker_color="#b8a9d9", **edge),
    go.Bar(name="Mycol processing", x=x, y=processing, offsetgroup=1, base=counting,
           marker_color="#6e5fa6",
           error_y=dict(type="data", array=[0, 0, 0, sd], visible=True,
                        color="black", thickness=1, width=6), **edge),
])

y_top = max(100.0, counting[-1] + processing[-1] + sd) + 8
fig_time.add_shape(type="line", x0=2.7, x1=3.3, y0=y_top, y1=y_top,
                   line=dict(color="black", width=1))
fig_time.add_annotation(x="Mean", y=y_top + 4, text=sig, showarrow=False, font=dict(size=12))
fig_time.update_layout(
    barmode="group", bargap=0.25, template="simple_white", width=750, height=500,
    yaxis_title="Percentage of Manual Counting Time", yaxis=dict(range=[0, y_top + 14]),
    font=dict(size=10),
    legend=dict(orientation="h", yanchor="top", y=0.99, xanchor="left", x=0.01,
                bgcolor="rgba(255,255,255,0.7)", bordercolor="lightgray", borderwidth=1),
)
fig_time.show()


## Per-image mean +/- SD, ranked by count

Sorted by mean count, so it shows whether variance grows with the count. The shared y axis lets the
error bars be compared between methods at matched count levels.


In [ ]:
METHODS, COUNTS = ("manual", "tool"), ("germinated", "ungerminated")

fig_rank = make_subplots(
    rows=2, cols=2, vertical_spacing=0.15, horizontal_spacing=0.08,
    subplot_titles=[f"{METHOD_LABEL[m]} - {c.capitalize()}" for m in METHODS for c in COUNTS])

y_max = 0
for i, method in enumerate(METHODS):
    for j, count in enumerate(COUNTS):
        per_image = (combined[combined["method"] == method]
                     .groupby("image_number")[count].agg(mean="mean", std="std")
                     .sort_values("mean", kind="stable").reset_index())
        y_max = max(y_max, float((per_image["mean"] + per_image["std"].fillna(0)).max()))
        fig_rank.add_trace(
            go.Scatter(x=np.arange(1, len(per_image) + 1), y=per_image["mean"],
                       error_y=dict(type="data", array=per_image["std"], visible=True,
                                    color="gray", thickness=1, width=2),
                       mode="markers", marker=dict(color="black", size=5),
                       name="Mean +/- SD", legendgroup="mean", showlegend=(i == 0 and j == 0),
                       customdata=per_image["image_number"],
                       hovertemplate="Image %{customdata}<br>mean: %{y:.2f}<extra></extra>"),
            row=i + 1, col=j + 1)

fig_rank.update_xaxes(title_text="Image (ordered by mean count)", showticklabels=False)
fig_rank.update_yaxes(title_text="Cell count", range=[0, y_max * 1.05], matches="y")
fig_rank.update_layout(height=750, width=1150, template="simple_white",
                       legend_title="", font=dict(size=14))
fig_rank.show()


## Export

Each figure as a PNG at 300 dpi. Laying the figure out in points (inches x 72) makes a `font.size` of
14 render as 14 pt; `scale = dpi/72` then lifts it to 300 dpi.


In [ ]:
from PIL import Image

DPI = 300


def save(fig, name, w_in, h_in, **layout):
    """Write one figure at w_in x h_in inches, 300 dpi. `layout` applies only to the
    exported copy, so the on-screen figure is left alone - the time chart's legend sits
    inside the plot on screen but reflows badly at the export size."""
    path = OUTPUT / f"{name}.png"
    go.Figure(fig).update_layout(**layout).write_image(
        path, width=w_in * 72, height=h_in * 72, scale=DPI / 72)
    with Image.open(path) as im:
        im.save(path, dpi=(DPI, DPI))
    print(f"wrote {path}  {path.stat().st_size:,} B")


save(fig_time, "time_comparison", 7.5, 5.0,
     legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
                 bgcolor="rgba(0,0,0,0)", bordercolor="rgba(0,0,0,0)", title=""))
save(fig_rank, "per_image_mean_sd", 6.25, 6.25)
